# Feature Engineering

This notebook transforms historical customer behaviour into customer-level features for churn prediction.

Only `eval_set == "prior"` order history is used for predictive feature construction. Held-out `train` and `test` orders are reserved for future churn-label construction and evaluation to prevent target leakage.

In [21]:
import pandas as pd
from pathlib import Path

In [22]:
RAW_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")

orders = pd.read_csv(
    RAW_DIR / "orders.csv",
    usecols=[
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "eval_set",
    ],
)

prior_orders = orders[
    orders["eval_set"] == "prior"
].copy()

print("All orders:", orders.shape)
print("Prior orders:", prior_orders.shape)
print("Prior customers:", prior_orders["user_id"].nunique())

All orders: (3421083, 7)
Prior orders: (3214874, 7)
Prior customers: 206209


## Customer Frequency Features

In [23]:
customer_frequency = (
    prior_orders
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "nunique"),
        max_order_number=("order_number", "max"),
    )
    .reset_index()
)

customer_frequency.head()

,user_id,total_orders,max_order_number
0,1,10,10
1,2,14,14
2,3,12,12
3,4,5,5
4,5,4,4


## Order Behaviour Features

In [24]:
customer_order_behaviour = (
    prior_orders
    .groupby("user_id")
    .agg(
        avg_days_between_orders=("days_since_prior_order", "mean"),
        latest_order_number=("order_number", "max"),
        latest_order_dow=("order_dow", "last"),
        latest_order_hour=("order_hour_of_day", "last"),
    )
    .reset_index()
)

customer_order_behaviour.head()

,user_id,avg_days_between_orders,latest_order_number,latest_order_dow,latest_order_hour
0,1,19.555556,10,4,8
1,2,15.230769,14,3,10
2,3,12.090909,12,1,15
3,4,13.750000,5,5,13
4,5,13.333333,4,1,18


## Purchase Cycle Features

In [25]:
purchase_cycle_features = (
    prior_orders
    .groupby("user_id")["days_since_prior_order"]
    .agg(
        median_purchase_interval="median",
        min_purchase_interval="min",
        max_purchase_interval="max",
        std_purchase_interval="std",
    )
    .reset_index()
)

purchase_cycle_features.head()

,user_id,median_purchase_interval,min_purchase_interval,max_purchase_interval,std_purchase_interval
0,1,20.0,0.0,30.0,9.395625
1,2,13.0,3.0,30.0,9.867065
2,3,11.0,7.0,21.0,5.375026
3,4,17.0,0.0,21.0,9.500000
4,5,11.0,10.0,19.0,4.932883


## Reorder Behaviour Features

Historical reorder behaviour is calculated from `order_products__prior` only.

In [26]:
orders_lookup = (
    prior_orders[["order_id", "user_id"]]
    .drop_duplicates("order_id")
    .set_index("order_id")
)

ORDER_PRODUCT_COLUMNS = [
    "order_id",
    "product_id",
    "reordered",
]

customer_reorder_chunks = []

for chunk in pd.read_csv(
    RAW_DIR / "order_products__prior.csv",
    usecols=ORDER_PRODUCT_COLUMNS,
    chunksize=200_000,
):
    chunk["user_id"] = chunk["order_id"].map(orders_lookup["user_id"])
    chunk = chunk.dropna(subset=["user_id"])

    customer_reorder_chunks.append(
        chunk.groupby("user_id").agg(
            prior_order_items=("product_id", "count"),
            prior_unique_products=("product_id", "nunique"),
            prior_reordered_items=("reordered", "sum"),
            prior_orders=("order_id", "nunique"),
        ).reset_index()
    )

customer_reorder = (
    pd.concat(customer_reorder_chunks, ignore_index=True)
    .groupby("user_id", as_index=False)
    .sum()
)

customer_reorder.head()

,user_id,prior_order_items,prior_unique_products,prior_reordered_items,prior_orders
0,1,59,57,41,10
1,2,195,192,93,14
2,3,88,88,55,12
3,4,18,18,1,5
4,5,37,37,14,4


In [27]:
customer_reorder["avg_items_per_order"] = (
    customer_reorder["prior_order_items"]
    / customer_reorder["prior_orders"].replace(0, pd.NA)
)

customer_reorder["reorder_rate"] = (
    customer_reorder["prior_reordered_items"]
    / customer_reorder["prior_order_items"].replace(0, pd.NA)
)

customer_reorder["products_per_order"] = (
    customer_reorder["prior_unique_products"]
    / customer_reorder["prior_orders"].replace(0, pd.NA)
)

customer_reorder.head()

,user_id,prior_order_items,prior_unique_products,prior_reordered_items,prior_orders,avg_items_per_order,reorder_rate,products_per_order
0,1,59,57,41,10,5.900000,0.694915,5.700000
1,2,195,192,93,14,13.928571,0.476923,13.714286
2,3,88,88,55,12,7.333333,0.625000,7.333333
3,4,18,18,1,5,3.600000,0.055556,3.600000
4,5,37,37,14,4,9.250000,0.378378,9.250000


## Product Diversity Features

Product diversity is calculated from historical prior-order purchases.

In [28]:
products = pd.read_csv(
    RAW_DIR / "products.csv",
    usecols=["product_id", "aisle_id", "department_id"],
)

product_category_map = products.set_index("product_id")[
    ["aisle_id", "department_id"]
]

category_chunk_counts = []

for chunk in pd.read_csv(
    RAW_DIR / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    chunksize=200_000,
):
    chunk["user_id"] = chunk["order_id"].map(orders_lookup["user_id"])
    chunk = chunk.dropna(subset=["user_id"])

    chunk = chunk.join(product_category_map, on="product_id")
    chunk = chunk.dropna(subset=["aisle_id", "department_id"])

    category_chunk_counts.append(
        chunk[["user_id", "aisle_id", "department_id"]]
        .drop_duplicates()
    )

customer_aisle_department = pd.concat(
    category_chunk_counts,
    ignore_index=True,
).drop_duplicates(
    ["user_id", "aisle_id", "department_id"]
)

product_diversity = (
    customer_aisle_department
    .groupby("user_id")
    .agg(
        unique_aisles=("aisle_id", "nunique"),
        unique_departments=("department_id", "nunique"),
    )
    .reset_index()
)

product_diversity.head()

,user_id,unique_aisles,unique_departments
0,1,12,7
1,2,33,13
2,3,16,9
3,4,14,9
4,5,16,9


## Product Category Behaviour Features

Category behaviour uses the product → aisle → department hierarchy and historical prior purchases only.

In [29]:
category_count_chunks = []

for chunk in pd.read_csv(
    RAW_DIR / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    chunksize=200_000,
):
    chunk["user_id"] = chunk["order_id"].map(orders_lookup["user_id"])
    chunk = chunk.dropna(subset=["user_id"])

    chunk = chunk.join(product_category_map, on="product_id")
    chunk = chunk.dropna(subset=["aisle_id", "department_id"])

    category_count_chunks.append(
        chunk.groupby(
            ["user_id", "aisle_id", "department_id"],
            as_index=False,
        ).size().rename(columns={"size": "purchase_count"})
    )

category_counts = (
    pd.concat(category_count_chunks, ignore_index=True)
    .groupby(
        ["user_id", "aisle_id", "department_id"],
        as_index=False,
    )["purchase_count"]
    .sum()
)

department_counts = (
    category_counts
    .groupby(["user_id", "department_id"], as_index=False)["purchase_count"]
    .sum()
)

department_totals = (
    department_counts.groupby("user_id")["purchase_count"].sum()
)

department_counts["department_share"] = (
    department_counts["purchase_count"]
    / department_counts["user_id"].map(department_totals)
)

dominant_department = (
    department_counts
    .groupby("user_id")["department_share"]
    .max()
    .rename("dominant_department_share")
    .reset_index()
)

aisle_counts = (
    category_counts
    .groupby(["user_id", "aisle_id"], as_index=False)["purchase_count"]
    .sum()
)

aisle_totals = aisle_counts.groupby("user_id")["purchase_count"].sum()

aisle_counts["aisle_share"] = (
    aisle_counts["purchase_count"]
    / aisle_counts["user_id"].map(aisle_totals)
)

dominant_aisle = (
    aisle_counts
    .groupby("user_id")["aisle_share"]
    .max()
    .rename("dominant_aisle_share")
    .reset_index()
)

category_behaviour = (
    dominant_department
    .merge(dominant_aisle, on="user_id", how="outer")
)

category_behaviour.head()

,user_id,dominant_department_share,dominant_aisle_share
0,1,0.372881,0.220339
1,2,0.246154,0.215385
2,3,0.431818,0.215909
3,4,0.166667,0.166667
4,5,0.513514,0.216216


## Recency / Inactivity Features

Absolute calendar recency cannot be calculated because the dataset does not provide order dates.

Therefore, these features represent historical inactivity behaviour using the customer's longest observed purchase interval relative to their typical purchase cycle. They must not be interpreted as current calendar-day recency.

In [30]:
inactivity_features = purchase_cycle_features[
    ["user_id", "median_purchase_interval", "max_purchase_interval"]
].copy()

inactivity_features["inactivity_gap"] = (
    inactivity_features["max_purchase_interval"]
    - inactivity_features["median_purchase_interval"]
)

inactivity_features["inactivity_ratio"] = (
    inactivity_features["max_purchase_interval"]
    / inactivity_features["median_purchase_interval"].replace(0, pd.NA)
)

inactivity_features.head()

,user_id,median_purchase_interval,max_purchase_interval,inactivity_gap,inactivity_ratio
0,1,20.0,30.0,10.0,1.5
1,2,13.0,30.0,17.0,2.307692
2,3,11.0,21.0,10.0,1.909091
3,4,17.0,21.0,4.0,1.235294
4,5,11.0,19.0,8.0,1.727273


## Feature Consolidation

All predictive features are consolidated into one row per customer.

No held-out `train` or `test` order information is used in this feature table.

In [31]:
features = (
    customer_frequency
    .merge(customer_order_behaviour, on="user_id", how="left", validate="one_to_one")
    .merge(purchase_cycle_features, on="user_id", how="left", validate="one_to_one")
    .merge(customer_reorder, on="user_id", how="left", validate="one_to_one")
    .merge(product_diversity, on="user_id", how="left", validate="one_to_one")
    .merge(category_behaviour, on="user_id", how="left", validate="one_to_one")
    .merge(
        inactivity_features[
            ["user_id", "inactivity_gap", "inactivity_ratio"]
        ],
        on="user_id",
        how="left",
        validate="one_to_one",
    )
)

print("Feature dataset shape:", features.shape)
print("Unique customers:", features["user_id"].nunique())
print("Duplicate user_id:", features["user_id"].duplicated().sum())

Feature dataset shape: (206209, 24)
Unique customers: 206209
Duplicate user_id: 0


In [32]:
print("Feature columns:")
for column in features.columns:
    print("-", column)

print("\nMissing values:")
print(features.isna().sum().sort_values(ascending=False).head(10))

Feature columns:
- user_id
- total_orders
- max_order_number
- avg_days_between_orders
- latest_order_number
- latest_order_dow
- latest_order_hour
- median_purchase_interval
- min_purchase_interval
- max_purchase_interval
- std_purchase_interval
- prior_order_items
- prior_unique_products
- prior_reordered_items
- prior_orders
- avg_items_per_order
- reorder_rate
- products_per_order
- unique_aisles
- unique_departments
- dominant_department_share
- dominant_aisle_share
- inactivity_gap
- inactivity_ratio

Missing values:
inactivity_ratio            106
user_id                       0
max_order_number              0
total_orders                  0
latest_order_number           0
latest_order_dow              0
latest_order_hour             0
avg_days_between_orders       0
median_purchase_interval      0
min_purchase_interval         0
dtype: int64


In [33]:
print(
    "Invalid reorder rates:",
    (
        (features["reorder_rate"] < 0)
        | (features["reorder_rate"] > 1)
    ).sum()
)

print(
    "Invalid dominant department shares:",
    (
        (features["dominant_department_share"] < 0)
        | (features["dominant_department_share"] > 1)
    ).sum()
)

print(
    "Invalid dominant aisle shares:",
    (
        (features["dominant_aisle_share"] < 0)
        | (features["dominant_aisle_share"] > 1)
    ).sum()
)

Invalid reorder rates: 0
Invalid dominant department shares: 0
Invalid dominant aisle shares: 0


In [34]:
output_path = INTERIM_DIR / "customer_features.csv"

features.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(features):,}")
print(f"Columns: {len(features.columns)}")

Saved: ..\data\interim\customer_features.csv
Rows: 206,209
Columns: 24
